# IT Support Dashboard - Supporting Notebook 1

This series of notebooks contains supporting documentation to the full report. Here, we provide quality checks for the full dataset, confirm whether the English-only sample is representitive of the full population available, and conduct text analytics.

## Cleaning Process

Notebook 1 provides the data cleaning process of the raw dataset. The following actions were taken:
- 7 Null values within Answers were removed
- 13 Data entry errors within the Tag columns were resolved
- Entries within the Priority and Language columnns were formatted
- Data types of all dimensions were amended
- Produced a clean document available as a CSV and Parquet
- Interim subsets were created to show which records were amended
- All output files were encoded in latin_1 to translate anomalous ascii characters to their intended input


### Importing

First, the required packages for this process were imported, the necessary folders were created using the config file, and the raw dataset was read.

In [25]:
# Importing essential packages
import sys
from pathlib import Path
import pandas as pd
import os

# Add parent directory (project root) to sys.path
project_root = Path(r"C:\Users\David\Desktop\Python_Files\IT-Support-Ticket-Analysis")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from config import Config

Config.ensure_directories()

# Reading the initial dataset as a DataFrame: df
df = pd.read_csv(
    Config.RAW_DATA_PATH,
    encoding="utf-8",
    engine="python",
    na_values=[
        "",
        " ",
        "NA",
        "N/A",
        "na",
        "n/a",
        "NULL",
        "null",
        "None",
        "none",
        "NAN",
        "NaN",
        "nan",
        None,
    ],
)

display(df)

[Config] Verified project directory structure under C:\Users\David\Desktop\Python_Files\IT-Support-Ticket-Analysis


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28582,Performance Problem with Data Analytics Tool,The data analytics tool experiences sluggish p...,We are addressing the performance issue with t...,Incident,Technical Support,high,en,400,Performance,IT,Tech Support,NaN,NaN,NaN,NaN,NaN
28583,Datensperrung in der Kundschaftsbetreuung,"Es gab einen Datensperrungsunfall, bei dem ung...",Ich kann Ihnen bei dem Datensperrungsunfall he...,Incident,Product Support,high,de,400,Security,IT,Tech Support,Bug,NaN,NaN,NaN,NaN
28584,Problem mit der Videokonferenz-Software heute,Wichtigere Sitzungen wurden unterbrochen durch...,"Sehr geehrte/r [Name], leider wurde das Proble...",Incident,Human Resources,low,de,400,Bug,Performance,Network,IT,Tech Support,NaN,NaN,NaN
28585,Update Request for SaaS Platform Integration F...,Requesting an update on the integration featur...,Received your request for updates on the integ...,Change,IT Support,high,en,400,Feature,IT,Tech Support,NaN,NaN,NaN,NaN,NaN


Second, we wanted to see how clean and complete this dataset was. We were looking at the shape of the table, any missing values, if dimensions were in appropriate datatypes, and how much memory they used.

In [26]:
# Information about the dataset
print("What is the shape of my table?")
print(df.shape)
print("\nAre there any missing values in each dimension?")
print(df.isna().sum().sort_values())
print("\nWhat is the datatype of each column?")
print(df.dtypes)
print("\nHow many bytes does each column use?")
print(df.memory_usage(deep=True))

What is the shape of my table?
(28587, 16)

Are there any missing values in each dimension?
body            0
type            0
queue           0
priority        0
language        0
version         0
tag_1           0
answer          7
tag_2          13
tag_3         136
tag_4        3058
subject      3838
tag_5       14042
tag_6       22713
tag_7       26547
tag_8       28022
dtype: int64

What is the datatype of each column?
subject     object
body        object
answer      object
type        object
queue       object
priority    object
language    object
version      int64
tag_1       object
tag_2       object
tag_3       object
tag_4       object
tag_5       object
tag_6       object
tag_7       object
tag_8       object
dtype: object

How many bytes does each column use?
Index            132
subject      2489723
body        12609156
answer      12608655
type         1609416
queue        1877997
priority     1532247
language     1457937
version       228696
tag_1        1604968
tag

### Removing null responses

The following columns contained nulls which would impede the text analytics pipeline planned later:
- 7 under the Answer column
- 3,838 under the Subject column

We removed these null values because we thought these would intefere with the text analytics planned later on, and were comfortable with losing 0.02% of total records. This was the simplest solution, as it wasn't possible to reconstruct the original answer provided by the Customer Service representative. 

In [27]:
# Identify rows with null answers; subjects and tags were ignored as they were less critical to this analysis
null_answers = df[df["answer"].isna()].copy()
null_answers = pd.concat([null_answers, df[df["subject"].isna()].copy()])

# Save the table of null answers to a CSV file
null_answers.to_csv(Config.NULL_ANSWERS_PATH, index=False, encoding="latin-1")

# Count the number of null answers
null_count = null_answers.shape[0]

# Perecnentage of null answers
percentage_null = (null_count / df.shape[0]) * 100

print(
    f"\nTable containing {null_count} rows ({percentage_null:.2f}%) with null answers:"
)
display(null_answers)


Table containing 3845 rows (13.45%) with null answers:


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
4956,Support Inquiry Regarding SendGrid Integration,"Dear Customer Service, I would like to inquire...",NaN,Request,Billing and Payments,high,de,52,Billing,Payment,Platform,Pricing,Discount,Promotion,Integration,Support
5381,Support Request for SendGrid Integration,"Dear Customer Service, I would like to inquire...",NaN,Request,Billing and Payments,high,de,52,Billing,Payment,Support,Integration,Pricing,Promotion,NaN,NaN
12163,Recent Decline in Engagement Metrics Noted Online,We have observed a decrease in engagement metr...,NaN,Problem,Product Support,medium,en,400,Feedback,Performance,Feature,NaN,NaN,NaN,NaN,NaN
13378,Recent Decrease in Engagement Metrics Noted On...,There has been a decline in engagement metrics...,NaN,Problem,Product Support,medium,en,400,Feedback,Performance,Feature,NaN,NaN,NaN,NaN,NaN
13651,NaN,We are sorry to hear that you are experiencing...,NaN,Problem,Technical Support,high,en,400,Bug,Performance,Feature,Documentation,Tech Support,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28550,NaN,"Dear Customer Support, I am writing to report ...",We appreciate you bringing this critical issue...,Incident,IT Support,medium,en,400,Security,Bug,Virus,Performance,NaN,NaN,NaN,NaN
28552,NaN,"geehrte Kundenservice, es wurde eine mögliche ...","geehrter [name], wir danken Ihnen für Ihre Anf...",Problem,Technical Support,high,de,400,Security,Bug,IT,Tech Support,NaN,NaN,NaN,NaN
28568,NaN,"Sehr geehrte Kundenservice, ich wende mich an ...","Wir bieten Social-Media-Management, Suchmaschi...",Request,IT Support,medium,de,400,Sales,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN
28569,NaN,"Sehr geehrte Kundensupport, ich begegne einem ...",Wir haben Ihre E-Mail über den Leistungsriss b...,Problem,Technical Support,low,de,400,Performance,Bug,Feedback,IT,Tech Support,NaN,NaN,NaN


In [28]:
# Dropping null answers and resetting index
df = df.dropna(subset=["answer"])
df = df.dropna(subset=["subject"])
df.reset_index(drop=True, inplace=True)

print("\nNumber of null answers after dropping:", df["answer"].isna().sum())


Number of null answers after dropping: 0


### Amending tags

Next, we corrected anomalies within the tags. Under the tag_1 column, 12 records were found with various tags stitched together using a comma. Seeing as there were no entries within the other tag columns, we assumed these were entered erroneously. These sequences of tags were separated and used to populate the other tag columns for each record.

Under the tag_3 column, 1 record was identified with a similar validation error. These tags were separated and used to populate succeeding field, shuffling the tags downward and maintaining the order of input.

No issues were found within the other tag columns.

In [29]:
# Number of tag columns
tag_columns = [col for col in df.columns if col.startswith("tag_")]
print(f"\nNumber of tag columns: {len(tag_columns)}")

long_tags_columns = []
index_list = []

# Loop to chechk for multiple tags
for i in range(1, len(tag_columns) + 1):
    tag_col = f"tag_{i}"
    long_tags = df[df[tag_col].str.contains(",", na=False)]

    if long_tags.empty:
        print(f"\nNo rows with multiple tags found in {tag_col}.")
    else:
        long_tags_columns.append(tag_col)
        long_tags_count = long_tags.shape[0]
        index_list.extend(df[df[tag_col].str.contains(",", na=False)].index.tolist())
        print(
            f"\nTable containing {long_tags_count} rows with multiple tags in {tag_col}:"
        )
        display(long_tags)

# Summary of columns with multiple tags
print(f"\nColumns {long_tags_columns} contain multiple tags separated by commas.")

# Saving the table of rows with multiple tags to a CSV file
df.loc[index_list].to_csv(Config.INVALID_TAGS_PATH, index=False, encoding="latin-1")


Number of tag columns: 8

Table containing 12 rows with multiple tags in tag_1:


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
948,Digitale Kampagnen erzielten unterdurchschnitt...,Die digitalen Kampagnen der Agentur schnitten ...,Überprüfen Sie die digitalen Kampagnen und ide...,Problem,Product Support,low,de,52,"Performance,Bug,Disruption,Security",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1704,Significant Underperformance in Today's Digita...,The marketing agency's digital advertising eff...,We received an email concerning the underperfo...,Problem,Technical Support,low,en,52,"Performance,Disruption,Outage,Monitoring,Analysis",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2036,Heute ist die Investmentplattform abgestürzt,"Sehr geehrter Kundendienst, ich möchte ein Pro...","<name>, vielen Dank, dass Sie das Problem unse...",Incident,Technical Support,high,de,52,"Crash,Performance,Outage,Disruption,Recovery,S...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2183,Significant Underperformance of Digital Campai...,The digital campaigns managed by the marketing...,We received your email regarding the underperf...,Incident,Technical Support,high,en,52,"Performance,Disruption,IT,Tech Support",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2256,Digitale Kampagnen erzielen plattformübergreif...,Die digitalen Kampagnen unserer Marketingagent...,"Die digitalen Kampagnen überprüfen, Telefonnum...",Problem,Product Support,high,de,52,"Performance,Disruption,Support",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2908,Digitale Marketingkampagne der Agentur zeigt h...,Die digitale Kampagne der Marketingagentur wei...,Ich habe eine E-Mail bezüglich der schlechten ...,Problem,Technical Support,low,de,52,"Performance,Outage,Disruption,Recovery,Marketi...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3246,Recent Unauthorized Access Attempts Detected o...,There have been recent attempts to gain unauth...,We received an email concerning unauthorized a...,Incident,Technical Support,medium,en,52,"Security,IT,Tech Support",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4275,Digitale Kampagnen schneiden plattformübergrei...,"Sehr geehrter Kundenservice, ich wende mich be...","<name>, ich helfe Ihnen gern bei der Fehlerbeh...",Problem,Technical Support,high,de,52,"Performance,Disruption,Outage,Support,Integration",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4774,Fortschrittliche Datenanalyse,Bitte nutzen Sie detaillierte Informationen zu...,"<name>, vielen Dank für Ihre Anfrage bezüglich...",Request,Technical Support,high,de,52,"Performance,Security,Feature,Documentation",NaN,NaN,NaN,NaN,NaN,NaN,NaN
5059,Hospital System Security Practices,Customer Support is inquiring about methods to...,"<name>, we understand your concerns regarding ...",Request,IT Support,high,en,52,"Security,IT,Tech Support,Data Privacy,Regulati...",NaN,NaN,NaN,NaN,NaN,NaN,NaN



No rows with multiple tags found in tag_2.

Table containing 1 rows with multiple tags in tag_3:


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
5210,Zugang zu medizinischen Daten,"Sehr geehrter Kundenservice, ich möchte Sie da...","Sehr geehrter <name>, wir nehmen die Sicherhei...",Problem,Technical Support,medium,de,52,Security,Network,"Disruption,IT",Tech Support,NaN,NaN,NaN,NaN



No rows with multiple tags found in tag_4.

No rows with multiple tags found in tag_5.

No rows with multiple tags found in tag_6.

No rows with multiple tags found in tag_7.

No rows with multiple tags found in tag_8.

Columns ['tag_1', 'tag_3'] contain multiple tags separated by commas.


In [30]:
# Creating a loop to populate the other tag columns with the additional tags from tag_1, then leaving the original tag_1 column with the first tag only
for col in long_tags_columns:
    if col == "tag_1":
        for i in index_list:
            if i == 5883:
                continue  # Skip the row with index 5883 as this needs to be transformed in the next loop
            row = df.loc[i]
            # print(f"Row {i}: {row['tag_1']}")
            tags = row["tag_1"].split(",")
            # Update tag_2, tag_3, tag_4 with additional tags
            df.loc[row.name, "tag_2"] = tags[1].strip() if len(tags) > 1 else None
            df.loc[row.name, "tag_3"] = tags[2].strip() if len(tags) > 2 else None
            df.loc[row.name, "tag_4"] = tags[3].strip() if len(tags) > 3 else None
            # Update tag_1 to only contain the first tag
            df.loc[row.name, "tag_1"] = tags[0].strip()

    if col == "tag_3":
        for i in df[df[col].str.contains(",", na=False)].index.tolist():
            row = df.loc[i]
            # print(f"Row {i}: {row['tag_3']}")
            tags = row["tag_3"].split(",")
            # Move tag_4 to tag_5, then update tag_4 with the second tag from tag_3
            df.loc[row.name, "tag_5"] = row["tag_4"]
            df.loc[row.name, "tag_4"] = tags[1].strip() if len(tags) > 1 else None
            # Update tag_3 to only contain the first tag
            df.loc[row.name, "tag_3"] = tags[0].strip()

# Checking the affected records to verify whether the changes were successfully implemented
print("\nUpdated rows with multiple tags:")
display(df[df.index.isin(index_list)])

# Saving the table of the records with amended tags to a CSV file
df.loc[index_list].to_csv(Config.VALIDATED_TAGS_PATH, index=False, encoding="latin-1")


Updated rows with multiple tags:


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
948,Digitale Kampagnen erzielten unterdurchschnitt...,Die digitalen Kampagnen der Agentur schnitten ...,Überprüfen Sie die digitalen Kampagnen und ide...,Problem,Product Support,low,de,52,Performance,Bug,Disruption,Security,NaN,NaN,NaN,NaN
1704,Significant Underperformance in Today's Digita...,The marketing agency's digital advertising eff...,We received an email concerning the underperfo...,Problem,Technical Support,low,en,52,Performance,Disruption,Outage,Monitoring,NaN,NaN,NaN,NaN
2036,Heute ist die Investmentplattform abgestürzt,"Sehr geehrter Kundendienst, ich möchte ein Pro...","<name>, vielen Dank, dass Sie das Problem unse...",Incident,Technical Support,high,de,52,Crash,Performance,Outage,Disruption,NaN,NaN,NaN,NaN
2183,Significant Underperformance of Digital Campai...,The digital campaigns managed by the marketing...,We received your email regarding the underperf...,Incident,Technical Support,high,en,52,Performance,Disruption,IT,Tech Support,NaN,NaN,NaN,NaN
2256,Digitale Kampagnen erzielen plattformübergreif...,Die digitalen Kampagnen unserer Marketingagent...,"Die digitalen Kampagnen überprüfen, Telefonnum...",Problem,Product Support,high,de,52,Performance,Disruption,Support,None,NaN,NaN,NaN,NaN
2908,Digitale Marketingkampagne der Agentur zeigt h...,Die digitale Kampagne der Marketingagentur wei...,Ich habe eine E-Mail bezüglich der schlechten ...,Problem,Technical Support,low,de,52,Performance,Outage,Disruption,Recovery,NaN,NaN,NaN,NaN
3246,Recent Unauthorized Access Attempts Detected o...,There have been recent attempts to gain unauth...,We received an email concerning unauthorized a...,Incident,Technical Support,medium,en,52,Security,IT,Tech Support,None,NaN,NaN,NaN,NaN
4275,Digitale Kampagnen schneiden plattformübergrei...,"Sehr geehrter Kundenservice, ich wende mich be...","<name>, ich helfe Ihnen gern bei der Fehlerbeh...",Problem,Technical Support,high,de,52,Performance,Disruption,Outage,Support,NaN,NaN,NaN,NaN
4774,Fortschrittliche Datenanalyse,Bitte nutzen Sie detaillierte Informationen zu...,"<name>, vielen Dank für Ihre Anfrage bezüglich...",Request,Technical Support,high,de,52,Performance,Security,Feature,Documentation,NaN,NaN,NaN,NaN
5059,Hospital System Security Practices,Customer Support is inquiring about methods to...,"<name>, we understand your concerns regarding ...",Request,IT Support,high,en,52,Security,IT,Tech Support,Data Privacy,NaN,NaN,NaN,NaN


### Standardising inputs

Then, we tidied up the dataset by removing variations within the Priority and Language columns, deleted formatting text relics from the Answer and Body dimensions, and changed the datatypes of all dimensions to reduce memory usage, improving performance downstream.

In [31]:
# Standardizing text in 'priority' and 'language' columns to title case
df["priority"] = df["priority"].str.title()
df["language"] = df["language"].str.upper()

# Removing text formatting from body and answer columns
columns_to_clean = ["answer", "body"]
for column in columns_to_clean:
    df[column] = df[column].str.replace(r" \n\n", " ")
    df[column] = df[column].str.replace(r"\n\n ", " ")
    df[column] = df[column].str.replace(r"\n\n", " ")
    df[column] = df[column].str.replace(r" \n", " ")
    df[column] = df[column].str.replace(r"\n ", " ")
    df[column] = df[column].str.replace(r"\n", " ")
    df[column] = df[column].str.replace(r" <br><br>", " ")
    df[column] = df[column].str.replace(r"<br><br> ", " ")
    df[column] = df[column].str.replace(r"<br><br>", " ")
    df[column] = df[column].str.replace(r" <br>", " ")
    df[column] = df[column].str.replace(r"<br> ", " ")
    df[column] = df[column].str.replace(r"<br>", " ")

# Capitalise the first letter of the headers of all columns
for col in df.columns:
    df.rename(columns={col: col.capitalize()}, inplace=True)

# Converting data types to save on memory usage
# Lists of data types by column
categories = [
    "Type",
    "Queue",
    "Priority",
    "Language",
    "Version",
    "Tag_1",
    "Tag_2",
    "Tag_3",
    "Tag_4",
    "Tag_5",
    "Tag_6",
    "Tag_7",
    "Tag_8",
]
strings = ["Subject", "Body", "Answer"]

# List comprehension loop
for types in df:
    # Categories
    for category in categories:
        if types == category:
            df[types] = df[types].astype("category")

    # Strings
    for string in strings:
        if types == string:
            df[types] = df[types].astype("string")

# Displaying the DataFrame after dropping null answers, resetting index, and converting data types
display(df)

,Subject,Body,Answer,Type,Queue,Priority,Language,Version,Tag_1,Tag_2,Tag_3,Tag_4,Tag_5,Tag_6,Tag_7,Tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team, ich möchte einen g...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,High,DE,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team, I am writing to re...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,High,EN,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team, I hope this messag...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,Medium,EN,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team, I hope this messag...",We appreciate you reaching out with your billi...,Request,Billing and Payments,Low,EN,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team, I hope this message reaches...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,Medium,EN,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24738,Performance Problem with Data Analytics Tool,The data analytics tool experiences sluggish p...,We are addressing the performance issue with t...,Incident,Technical Support,High,EN,400,Performance,IT,Tech Support,NaN,NaN,NaN,NaN,NaN
24739,Datensperrung in der Kundschaftsbetreuung,"Es gab einen Datensperrungsunfall, bei dem ung...",Ich kann Ihnen bei dem Datensperrungsunfall he...,Incident,Product Support,High,DE,400,Security,IT,Tech Support,Bug,NaN,NaN,NaN,NaN
24740,Problem mit der Videokonferenz-Software heute,Wichtigere Sitzungen wurden unterbrochen durch...,"Sehr geehrte/r [Name], leider wurde das Proble...",Incident,Human Resources,Low,DE,400,Bug,Performance,Network,IT,Tech Support,NaN,NaN,NaN
24741,Update Request for SaaS Platform Integration F...,Requesting an update on the integration featur...,Received your request for updates on the integ...,Change,IT Support,High,EN,400,Feature,IT,Tech Support,NaN,NaN,NaN,NaN,NaN


### Validation

The dataset was revalidated to confirm whether all changes had successfully been applied.

In [32]:
print("What is the shape of my table?")
print(df.shape)
print("\nAre there any missing values in each dimension?")
print(df.isna().sum().sort_values())
print("\nWhat is the datatype of each column?")
print(df.dtypes)
print("\nHow many bytes does each column use?")
print(df.memory_usage(deep=True))

What is the shape of my table?
(24743, 16)

Are there any missing values in each dimension?
Subject         0
Body            0
Answer          0
Type            0
Queue           0
Priority        0
Language        0
Version         0
Tag_1           0
Tag_2           1
Tag_3          97
Tag_4        2630
Tag_5       12329
Tag_6       19883
Tag_7       23150
Tag_8       24333
dtype: int64

What is the datatype of each column?
Subject     string[python]
Body        string[python]
Answer      string[python]
Type              category
Queue             category
Priority          category
Language          category
Version           category
Tag_1             category
Tag_2             category
Tag_3             category
Tag_4             category
Tag_5             category
Tag_6             category
Tag_7             category
Tag_8             category
dtype: object

How many bytes does each column use?
Index            132
Subject      2366005
Body        10682089
Answer      10839569
T

To close off the validation, we looked at how many unique values were found under each dimension within this dataset.

In [33]:
# Loop to extract all unique values from each column in df
for column in df.columns:
    unique_values = df[column].sort_values(ascending=True).unique()
    length = len(unique_values)
    print(f"There were {length} unique values in {column}:\n{unique_values}")
    print("")

There were 24743 unique values in Subject:
<StringArray>
[                                                                                                                                                                                              ' Assistance Request',
                                                                                                                                         ' Bitte um Ausführliche Informationen zur Datenaufbereitungsdienstleistung',
                                                                                                                                                                   ' Datenschutzverletzung in Krankenhaus-Systemen ',
                                                                                                                                                                                ' Reported Problem with Data Access',
                                                                                       

### Saving the clean dataset

Two versions were created, a CSV and a Parquet, for downstream use. The CSV version was made to quickly and easily accessing the data, while the Parquet file was for faster opening and reading of the data in later analyses.

In [34]:
# CSV – portable
df.to_csv(Config.CLEAN_CSV_PATH, index=False, encoding="latin-1")

# Parquet – efficient
df.to_parquet(
    Config.CLEAN_PARQUET_PATH, index=False, engine="pyarrow", compression="snappy"
)

# Summary of saved files
for path in [Config.CLEAN_CSV_PATH, Config.CLEAN_PARQUET_PATH]:
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"{path.name:<25} | {size_mb:>6.2f} MB")

print("\nAll files saved successfully.")

04_Tickets_Clean.csv      |  21.19 MB
04_Tickets_Clean.parquet  |   9.72 MB

All files saved successfully.
